# M04 - Training a model from scratch

**Hardware:** CPU (under a minute) · **Time:** 4-6 hours ·
**Prerequisite:** M02-M03, basic Python

This notebook trains a small convolutional network on the synthetic shape
dataset that the computer-vision labs use. It covers the parts of a real
training pipeline that a model summary hides: data audit, grouped splits,
augmentation, learning-rate choice, per-class evaluation, checkpointing, and a
measured mixed-precision check.

The most important result is not the accuracy. It is the gap between a random
split and a grouped split. When near-duplicate images sit on both sides, the
test score measures memorisation, not recognition.

**You will produce:** a leakage audit, a trained classifier, a metric table,
and a checkpoint.

## 1. Setup and the curated dataset

`vision_core.dataio` provides the shared dataset used by the CV labs: 100
shapes, each rendered three times with small shifts and brightness changes.
The variants share a `source_id`, like burst photos or consecutive video
frames. The curated train/validation/test arrays are split by source, so a
source never appears in two splits.

The labels are `circle`, `square`, and `triangle`, 32x32 pixels, CC0-1.0.

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "projects").is_dir())
sys.path.insert(0, str(ROOT / "projects" / "computer-vision"))

from vision_core import dataio, datasets, evaluation, imaging, models, training

print("torch", torch.__version__, "| device", "cuda" if torch.cuda.is_available() else "cpu")
models.set_seed(0)

splits = dataio.classification_splits()
records = dataio.classification_records()
class_names = list(datasets.SHAPE_CLASSES)
counts = {split: len(part["labels"]) for split, part in splits.items()}
print("split sizes:", counts, "| total images:", sum(counts.values()))
print("sources:", len({record.source_id for record in records}))
print("class balance:", datasets.class_balance(records, class_names))
print("dataset card:", dataio.dataset_card()["licence"])
assert counts["train"] > counts["test"] > 0
assert len(records) == 300

torch 2.7.1+cpu | device cpu
split sizes: {'train': 210, 'val': 45, 'test': 45} | total images: 300
sources: 100
class balance: {'circle': 102, 'square': 99, 'triangle': 99}
dataset card: CC0-1.0 (public domain dedication)


## 2. Two ways to split, one honest number

`random_split` shuffles images. `group_split` shuffles **sources** and keeps all
variants of a source together. Both use the same fractions here, so any
difference in the score comes from the split, not from the data.

`audit_leakage` reports a **hard leak** (the same source in two splits, which
invalidates the test score) and a **soft leak** (different sources that look
nearly identical, a genuine ambiguity).

In [2]:
fractions = (0.7, 0.15, 0.15)
random_splits = datasets.random_split(records, fractions, seed=0)
group_splits = datasets.group_split(records, fractions, seed=0)

for name, candidate in [("random", random_splits), ("grouped", group_splits)]:
    audit = datasets.audit_leakage(candidate)
    sizes = {split: len(items) for split, items in candidate.items()}
    print(f"{name:8s} sizes={sizes} hard_leak={audit['hard_leaky']} "
          f"shared_sources={audit['n_shared_sources']} near_duplicates={audit['cross_split_near_duplicate_count']}")

assert datasets.audit_leakage(random_splits)["hard_leaky"] is True
assert datasets.audit_leakage(group_splits)["hard_leaky"] is False

random   sizes={'train': 210, 'val': 45, 'test': 45} hard_leak=True shared_sources=91 near_duplicates=14


grouped  sizes={'train': 210, 'val': 45, 'test': 45} hard_leak=False shared_sources=0 near_duplicates=1


In [3]:
def arrays(part):
    return np.stack([record.image for record in part]), np.array([record.label for record in part])


def train_and_score(candidate, epochs=8, seed=0):
    train_x, train_y = arrays(candidate["train"])
    val_x, val_y = arrays(candidate["val"])
    test_x, test_y = arrays(candidate["test"])
    models.set_seed(seed)
    model = models.TinyCNN(num_classes=len(class_names), width=16, dropout=0.1)
    history, model = training.train_classifier(
        model, train_x, train_y, val_x, val_y,
        epochs=epochs, batch_size=16, lr=2e-3, seed=seed, log=False,
    )
    mean, std = training.dataset_stats(train_x)
    predicted, _ = training.predict(model, test_x, mean, std)
    return float((predicted == test_y).mean()), history, (mean, std)


random_acc, random_history, _ = train_and_score(random_splits)
group_acc, group_history, stats = train_and_score(group_splits)
print("random-split test accuracy:", round(random_acc, 3))
print("grouped-split test accuracy:", round(group_acc, 3))
print("leakage premium:", round(random_acc - group_acc, 3))

random-split test accuracy: 0.867
grouped-split test accuracy: 0.622
leakage premium: 0.244


## 3. What a convolution actually computes

A 3x3 kernel slides over the image; each output pixel is a dot product of the
kernel with a 3x3 patch. The edge kernel below responds to horizontal changes.
The network stacks these operations and learns the kernels instead of using
hand-written ones.

**Receptive field** is how much of the input one feature sees. Each 3x3
convolution adds 2 pixels; each 2x2 max-pool doubles the stride. `TinyCNN` uses
three blocks of `conv-conv-pool`, so the final feature sees 29 of the 32 pixels
- which is why global average pooling can summarise the image.

In [4]:
sample = records[0].image.astype(np.float32) / 255.0
edge_kernel = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=np.float32)

def convolve(image, kernel):
    height, width = image.shape[:2]
    out = np.zeros((height - 2, width - 2), dtype=np.float32)
    for row in range(out.shape[0]):
        for col in range(out.shape[1]):
            out[row, col] = float((image[row : row + 3, col : col + 3] * kernel).sum())
    return out

response = convolve(sample.mean(axis=2), edge_kernel)
print("input", sample.shape, "-> response", response.shape)
print("response range:", round(float(response.min()), 3), "to", round(float(response.max()), 3))

receptive, jump = 1, 1
for _ in range(3):
    for _ in range(2):
        receptive += 2 * jump
    jump *= 2
print("receptive field after three conv-conv-pool blocks:", receptive, "pixels")
assert response.shape == (30, 30)

input (32, 32, 3) -> response (30, 30)
response range: -0.216 to 0.216
receptive field after three conv-conv-pool blocks: 29 pixels


## 4. The model, plain and augmented

`TinyCNN` is three convolutional blocks followed by global average pooling and a
linear classifier. Global pooling keeps the parameter count independent of the
input size.

Augmentation uses the shared `AugmentConfig` from `vision_core.imaging`:
horizontal flip, a one-pixel shift, and a small brightness change. The
transformations are label-preserving for shapes. Both models get the same
25-epoch budget, so the comparison is not confounded by extra training time.
Augmentation is not free accuracy: flipping a traffic sign or mirroring text
changes the label, which is why every transformation needs a
label-preservation argument.

In [5]:
train_x, train_y = splits["train"]["images"], splits["train"]["labels"]
val_x, val_y = splits["val"]["images"], splits["val"]["labels"]
test_x, test_y = splits["test"]["images"], splits["test"]["labels"]
mean, std = training.dataset_stats(train_x)

models.set_seed(0)
plain = models.TinyCNN(num_classes=len(class_names), width=16, dropout=0.1)
started = time.perf_counter()
plain_history, plain = training.train_classifier(
    plain, train_x, train_y, val_x, val_y,
    epochs=25, batch_size=16, lr=2e-3, seed=0, log=False,
)
plain_seconds = time.perf_counter() - started
print(plain_history.ascii_plot("val_acc"))
plain_pred, _ = training.predict(plain, test_x, mean, std)
plain_acc = float((plain_pred == test_y).mean())
print("plain test accuracy:", round(plain_acc, 3), "| best val", round(plain_history.best_val_acc, 3),
      "| seconds", round(plain_seconds, 2))
print("parameters:", models.count_parameters(plain), "| disk MB:", round(models.model_size_mb(plain), 3))

 1.000 |                 ##      
 0.931 |         #      #     ###
 0.861 |              #    ###   
 0.792 |            #            
 0.722 |       ##  # # #         
 0.653 |     #                   
 0.583 |  #   #   #              
 0.514 |    #                    
 0.444 |## #                     
       +-------------------------
plain test accuracy: 0.956 | best val 1.0 | seconds 2.57
parameters: 72499 | disk MB: 0.29


In [6]:
aug_config = imaging.AugmentConfig(hflip_p=0.5, max_shift=1, brightness=0.1)
augment_fn = lambda image, rng: imaging.augment(image, aug_config, rng)

models.set_seed(0)
augmented = models.TinyCNN(num_classes=len(class_names), width=16, dropout=0.1)
started = time.perf_counter()
aug_history, augmented = training.train_classifier(
    augmented, train_x, train_y, val_x, val_y,
    epochs=25, batch_size=16, lr=2e-3, seed=0, augment_fn=augment_fn, log=False,
)
aug_seconds = time.perf_counter() - started
aug_pred, _ = training.predict(augmented, test_x, mean, std)
aug_acc = float((aug_pred == test_y).mean())
print("augmented test accuracy:", round(aug_acc, 3), "| best val", round(aug_history.best_val_acc, 3),
      "| seconds", round(aug_seconds, 2))
print("overfitting gap: plain", round(plain_history.overfitting_gap(), 3),
      "| augmented", round(aug_history.overfitting_gap(), 3))

augmented test accuracy: 1.0 | best val 1.0 | seconds 3.11
overfitting gap: plain 0.067 | augmented 0.017


## 5. Learning rate is the highest-leverage hyperparameter

Three short runs, same data and seed, different learning rates. Too small and
the model has not converged within the budget; too large and the loss bounces.
This is a search, not a guess, and at this scale it costs seconds.

In [7]:
print("lr       val_acc  train_loss")
for lr in (5e-3, 2e-3, 2e-4):
    models.set_seed(0)
    candidate = models.TinyCNN(num_classes=len(class_names), width=16, dropout=0.1)
    history, candidate = training.train_classifier(
        candidate, train_x, train_y, val_x, val_y,
        epochs=4, batch_size=16, lr=lr, seed=0, log=False,
    )
    print(f"{lr:<8} {history.best_val_acc:.3f}    {history.train_loss[-1]:.3f}")

lr       val_acc  train_loss


0.005    0.422    1.094


0.002    0.578    0.786


0.0002   0.622    0.781


## 6. Per-class evaluation

Overall accuracy hides which class fails. `error_analysis` prints a per-class
report, the most confused class pairs, and the most **confident** errors. A
wrong prediction at 0.99 confidence is a different bug from a wrong prediction
at 0.36: the first needs data or labels, the second may need calibration.

The augmented model happens to make no test errors on this small split, so the
report below runs on the plain model. A clean per-class table on a perfect model
tells you nothing about failure modes.

In [8]:
analysis = evaluation.error_analysis(
    plain, test_x, test_y, class_names, mean=mean, std=std
)
print(analysis["report"])
print("confusion pairs:", analysis["confusion_pairs"])
print("errors:", analysis["n_errors"], "of", analysis["n_samples"])
for case in analysis["most_confident_errors"][:5]:
    print("  ", class_names[case["true"]], "->", class_names[case["predicted"]],
          "confidence", case["confidence"])

{'accuracy': 0.9555555555555556, 'macro_f1': 0.9586466165413534, 'weighted_f1': 0.955137844611529, 'per_class': {'circle': {'precision': 1.0, 'recall': 0.8666666666666667, 'f1': 0.9285714285714286, 'support': 15}, 'square': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 12}, 'triangle': {'precision': 0.9, 'recall': 1.0, 'f1': 0.9473684210526316, 'support': 18}}, 'confusion_matrix': [[13, 0, 2], [0, 12, 0], [0, 0, 18]]}
confusion pairs: {'circle -> triangle': 2}
errors: 2 of 45
   circle -> triangle confidence 0.5633
   circle -> triangle confidence 0.4877


## 7. Checkpoints and the compute budget

A checkpoint is weights plus the metadata needed to interpret them: class names,
normalisation statistics, input size, and a measured score. Without the
normalisation statistics the same weights produce different predictions - one of
the most common deployment bugs.

Mixed precision is a compute decision, not a quality decision. On CPU, bfloat16
autocast can be slower than float32 because the hardware lacks fast bf16
kernels. We measure it instead of assuming. On a supported GPU the same block
typically reduces memory and increases throughput.

In [9]:
checkpoint_path = ROOT / "notebooks" / "model-training" / "outputs" / "m04_tinycnn.pt"
meta = {
    "classes": class_names,
    "mean": mean,
    "std": std,
    "input_size": 32,
    "test_accuracy": round(aug_acc, 4),
    "epochs": 25,
    "augmented": True,
}
models.save_checkpoint(augmented, checkpoint_path, meta)
restored = models.TinyCNN(num_classes=len(class_names), width=16, dropout=0.1)
loaded_meta = models.load_checkpoint(restored, checkpoint_path)
restored_pred, _ = training.predict(restored, test_x, loaded_meta["mean"], loaded_meta["std"])
print("saved", checkpoint_path.name, round(models.model_size_mb(augmented), 3), "MB")
print("metadata keys:", sorted(loaded_meta))
print("identical predictions after reload:", bool(np.array_equal(restored_pred, aug_pred)))
assert np.array_equal(restored_pred, aug_pred)

saved m04_tinycnn.pt 0.29 MB
metadata keys: ['augmented', 'classes', 'epochs', 'input_size', 'mean', 'std', 'test_accuracy']
identical predictions after reload: True


In [10]:
batch = training.prepare_inputs(test_x[:16], mean, std)

def bench(dtype, iterations=10):
    augmented.eval()
    with torch.no_grad():
        for _ in range(2):
            with torch.autocast(device_type="cpu", dtype=dtype, enabled=dtype is not None):
                augmented(batch)
        started = time.perf_counter()
        for _ in range(iterations):
            with torch.autocast(device_type="cpu", dtype=dtype, enabled=dtype is not None):
                augmented(batch)
        return (time.perf_counter() - started) / iterations * 1000

with torch.no_grad():
    fp32_logits = augmented(batch)
try:
    with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
        bf16_logits = augmented(batch)
    max_diff = float((fp32_logits - bf16_logits.float()).abs().max())
    fp32_ms, bf16_ms = bench(None), bench(torch.bfloat16)
    print(f"fp32 {fp32_ms:.2f} ms/batch | bf16 {bf16_ms:.2f} ms/batch | max logit diff {max_diff:.5f}")
    print("speed ratio (fp32/bf16):", round(fp32_ms / bf16_ms, 2))
except RuntimeError as error:
    print("bfloat16 autocast unavailable on this CPU:", error)
    bf16_ms = None

fp32 1.68 ms/batch | bf16 3.56 ms/batch | max logit diff 0.01791
speed ratio (fp32/bf16): 0.47


## 8. Validation and exercises

**Exercises**

1. Use 20 sources instead of 100. How does the grouped test score change? Does
   the leakage premium grow or shrink?
2. Replace the edge kernel with a blur kernel. Predict the response image before
   running the code, then check your prediction.
3. Remove global average pooling in favour of flattening. Compute the parameter
   count before and after. Which version is invariant to input size?
4. Train with `patience=2`. Compare the best epoch with the final epoch. Why is
   returning the best weights not cheating?
5. Save two checkpoints with different normalisation statistics and compare
   predictions on the same image. Explain the failure mode in one sentence.

In [11]:
checks = {
    "curated dataset loaded": sum(counts.values()) == 300,
    "random split leaks sources": datasets.audit_leakage(random_splits)["hard_leaky"],
    "grouped split has no source leak": not datasets.audit_leakage(group_splits)["hard_leaky"],
    "convolution shape": response.shape == (30, 30),
    "plain model learned": plain_acc > 0.6,
    "augmented model learned": aug_acc > 0.6,
    "checkpoint reload is exact": np.array_equal(restored_pred, aug_pred),
    "per-class report covers test set": analysis["n_samples"] == len(test_y),
}
for name, passed in checks.items():
    print(f"{'ok' if passed else 'FAIL'} - {name}")
    assert passed, name
print("\nM04 validation passed.")

ok - curated dataset loaded
ok - random split leaks sources
ok - grouped split has no source leak
ok - convolution shape
ok - plain model learned
ok - augmented model learned
ok - checkpoint reload is exact
ok - per-class report covers test set

M04 validation passed.
